# 📈 Ridge & Lasso Regression — Solutions Notebook

**Complete, verified solutions.** Try the practice notebook first!

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

## 🔧 Section 3: Implementation from Scratch

### 3.1 Generate Data

In [ ]:
np.random.seed(42)
m = 200
n_features = 10

X = np.random.randn(m, n_features)
true_theta = np.array([5.0, -3.0, 2.0, 0, 0, 0, 0, 0, 0, 0])
y = X @ true_theta + 3.0 + np.random.randn(m) * 0.5

split = int(0.8 * m)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f'X shape: {X.shape}, True coefs: {true_theta}')

### 3.2 Ridge Regression from Scratch

In [ ]:
# ✅ SOLUTION: Ridge from scratch
def ridge_regression(X, y, alpha):
    """
    Ridge regression: θ = (X^T X + αI)^(-1) X^T y
    """
    # Center data to handle intercept separately
    X_mean = X.mean(axis=0)
    y_mean = y.mean()
    X_c = X - X_mean
    y_c = y - y_mean
    
    n = X.shape[1]
    I = np.eye(n)
    theta = np.linalg.inv(X_c.T @ X_c + alpha * I) @ X_c.T @ y_c
    intercept = y_mean - X_mean @ theta
    
    return theta, intercept


theta_ridge, intercept_ridge = ridge_regression(X_train, y_train, alpha=1.0)

print(f'Ridge (α=1.0) coefficients:')
for i, (t, true_t) in enumerate(zip(theta_ridge, true_theta)):
    print(f'  θ{i} = {t:7.4f}  (true: {true_t:.1f})')
print(f'  intercept = {intercept_ridge:.4f}  (true: 3.0)')

# Verify on test set
y_pred_ridge = X_test @ theta_ridge + intercept_ridge
r2_ridge = r2_score(y_test, y_pred_ridge)
print(f'\n  Test R² = {r2_ridge:.4f}')
print('\n✅ Ridge works! Note: irrelevant features are shrunk but NOT zero.')

### 3.3 Lasso from Scratch (Coordinate Descent)

In [ ]:
# ✅ SOLUTION: Lasso with coordinate descent
def soft_threshold(rho, alpha):
    """Soft thresholding operator."""
    if rho > alpha:
        return rho - alpha
    elif rho < -alpha:
        return rho + alpha
    else:
        return 0.0


def lasso_coordinate_descent(X, y, alpha, num_iters=1000, tol=1e-6):
    """
    Lasso regression using coordinate descent.
    """
    m, n = X.shape
    
    # Center data
    X_mean = X.mean(axis=0)
    y_mean = y.mean()
    X_c = X - X_mean
    y_c = y - y_mean
    
    theta = np.zeros(n)
    
    for iteration in range(num_iters):
        theta_old = theta.copy()
        
        for j in range(n):
            # Compute residual without feature j
            r_j = y_c - X_c @ theta + X_c[:, j] * theta[j]
            
            # Compute update
            rho_j = X_c[:, j] @ r_j / m
            z_j = X_c[:, j] @ X_c[:, j] / m
            
            # Apply soft thresholding
            theta[j] = soft_threshold(rho_j, alpha) / z_j
        
        # Check convergence
        if np.max(np.abs(theta - theta_old)) < tol:
            break
    
    intercept = y_mean - X_mean @ theta
    return theta, intercept


theta_lasso, intercept_lasso = lasso_coordinate_descent(X_train, y_train, alpha=0.1)

print(f'Lasso (α=0.1) coefficients:')
for i, (t, true_t) in enumerate(zip(theta_lasso, true_theta)):
    zero_marker = ' ← ZEROED OUT ✓' if abs(t) < 1e-6 else ''
    print(f'  θ{i} = {t:7.4f}  (true: {true_t:.1f}){zero_marker}')
print(f'  intercept = {intercept_lasso:.4f}  (true: 3.0)')

n_zero = sum(abs(t) < 1e-6 for t in theta_lasso)
print(f'\n  # Zero coefficients: {n_zero}/10')

y_pred_lasso = X_test @ theta_lasso + intercept_lasso
r2_lasso = r2_score(y_test, y_pred_lasso)
print(f'  Test R² = {r2_lasso:.4f}')
print('\n✅ Lasso correctly zeros out irrelevant features!')

---
## 📦 Section 4: Using scikit-learn

In [ ]:
# ✅ SOLUTION: Compare all models with sklearn
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

models = {
    'Linear (no reg)': LinearRegression(),
    'Ridge (α=1.0)': Ridge(alpha=1.0),
    'Lasso (α=0.1)': Lasso(alpha=0.1),
    'ElasticNet (α=0.1)': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

print(f'{"Model":<22} {"R²":>8} {"RMSE":>8} {"# Zeros":>8}')
print('-' * 50)

for name, model in models.items():
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    n_zeros = sum(abs(c) < 1e-6 for c in model.coef_)
    print(f'{name:<22} {r2:8.4f} {rmse:8.4f} {n_zeros:8d}')

print('\nCoefficients comparison:')
print(f'{"Feature":>10} {"True":>8} {"Linear":>8} {"Ridge":>8} {"Lasso":>8} {"ENet":>8}')
print('-' * 60)
model_list = list(models.values())
for i in range(n_features):
    print(f'{i:>10} {true_theta[i]:8.2f} '
          f'{model_list[0].coef_[i]:8.4f} {model_list[1].coef_[i]:8.4f} '
          f'{model_list[2].coef_[i]:8.4f} {model_list[3].coef_[i]:8.4f}')

### 4.1 Automatic Alpha Selection

In [ ]:
# ✅ SOLUTION: Auto alpha with CV
alphas_range = np.logspace(-4, 4, 50)

ridge_cv = RidgeCV(alphas=alphas_range, cv=5)
ridge_cv.fit(X_train_s, y_train)

lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train_s, y_train)

enet_cv = ElasticNetCV(cv=5, l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], random_state=42)
enet_cv.fit(X_train_s, y_train)

print('Automatic Alpha Selection Results:')
print(f'  RidgeCV:      best α = {ridge_cv.alpha_:.6f}, '
      f'R² = {ridge_cv.score(X_test_s, y_test):.4f}')
print(f'  LassoCV:      best α = {lasso_cv.alpha_:.6f}, '
      f'R² = {lasso_cv.score(X_test_s, y_test):.4f}, '
      f'zeros = {sum(abs(c) < 1e-6 for c in lasso_cv.coef_)}')
print(f'  ElasticNetCV: best α = {enet_cv.alpha_:.6f}, '
      f'l1_ratio = {enet_cv.l1_ratio_:.1f}, '
      f'R² = {enet_cv.score(X_test_s, y_test):.4f}')

---
## 🧪 Section 5: Experiments

### 5.1 Coefficient Paths

In [ ]:
# ✅ SOLUTION: Coefficient paths
alphas_path = np.logspace(-2, 4, 100)

ridge_coefs = []
lasso_coefs = []

for a in alphas_path:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_s, y_train)
    ridge_coefs.append(ridge.coef_.copy())
    
    lasso = Lasso(alpha=a, max_iter=10000)
    lasso.fit(X_train_s, y_train)
    lasso_coefs.append(lasso.coef_.copy())

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Ridge path
for i in range(n_features):
    label = f'θ{i} (true={true_theta[i]:.0f})'
    lw = 2.5 if true_theta[i] != 0 else 1
    alpha_val = 1.0 if true_theta[i] != 0 else 0.4
    axes[0].plot(alphas_path, ridge_coefs[:, i], linewidth=lw, alpha=alpha_val, label=label)
axes[0].set_xscale('log')
axes[0].set_xlabel('α (regularization strength)', fontsize=12)
axes[0].set_ylabel('Coefficient value', fontsize=12)
axes[0].set_title('Ridge Coefficient Path', fontsize=14)
axes[0].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[0].legend(fontsize=8, loc='upper right')

# Lasso path
for i in range(n_features):
    label = f'θ{i} (true={true_theta[i]:.0f})'
    lw = 2.5 if true_theta[i] != 0 else 1
    alpha_val = 1.0 if true_theta[i] != 0 else 0.4
    axes[1].plot(alphas_path, lasso_coefs[:, i], linewidth=lw, alpha=alpha_val, label=label)
axes[1].set_xscale('log')
axes[1].set_xlabel('α (regularization strength)', fontsize=12)
axes[1].set_ylabel('Coefficient value', fontsize=12)
axes[1].set_title('Lasso Coefficient Path', fontsize=14)
axes[1].axhline(y=0, color='k', linestyle='-', linewidth=0.5)
axes[1].legend(fontsize=8, loc='upper right')

plt.suptitle('Regularization Paths: Ridge vs Lasso', fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

print('Key observation:')
print('• Ridge: ALL coefficients shrink toward 0 but NEVER reach 0')
print('• Lasso: Irrelevant coefficients HIT 0 as α increases → feature selection!')

### 5.2 Ridge vs Lasso on High-Degree Polynomial

In [ ]:
# ✅ SOLUTION: Taming overfitting with regularization
np.random.seed(42)
X_nl = np.sort(np.random.uniform(-3, 3, 50)).reshape(-1, 1)
y_nl = 1 + 0.5 * X_nl.ravel() + 2 * X_nl.ravel()**2 + np.random.randn(50) * 2

# Create degree-15 features
poly = PolynomialFeatures(degree=15, include_bias=False)
scaler_nl = StandardScaler()

X_poly = poly.fit_transform(X_nl)
X_poly_s = scaler_nl.fit_transform(X_poly)

# Fit models
lr = LinearRegression().fit(X_poly_s, y_nl)
ridge_nl = Ridge(alpha=1.0).fit(X_poly_s, y_nl)
lasso_nl = Lasso(alpha=0.5, max_iter=10000).fit(X_poly_s, y_nl)

# Plot
X_plot = np.linspace(-3, 3, 300).reshape(-1, 1)
X_plot_poly = scaler_nl.transform(poly.transform(X_plot))

plt.figure(figsize=(14, 6))
plt.scatter(X_nl, y_nl, alpha=0.7, edgecolors='k', linewidth=0.3, s=40, zorder=5, label='Data')
plt.plot(X_plot, lr.predict(X_plot_poly), 'r-', linewidth=2, alpha=0.7, label='No regularization')
plt.plot(X_plot, ridge_nl.predict(X_plot_poly), 'b-', linewidth=2.5, label='Ridge (α=1.0)')
plt.plot(X_plot, lasso_nl.predict(X_plot_poly), 'g--', linewidth=2.5, label='Lasso (α=0.5)')

plt.xlabel('X', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.title('Degree-15 Polynomial: Regularization Saves the Day!', fontsize=14)
plt.legend(fontsize=11)
plt.ylim(-10, 40)
plt.show()

print('Observation: Without regularization, degree-15 polynomial overfits wildly.')
print('Ridge and Lasso smooth the fit, with Lasso being sparser.')
print(f'\nLasso used {sum(abs(c) > 1e-6 for c in lasso_nl.coef_)}/15 polynomial terms')

### 5.3 Multicollinearity Handling

In [ ]:
# ✅ SOLUTION: Ridge vs OLS with multicollinearity
np.random.seed(42)
m_col = 100
x1 = np.random.randn(m_col)
x2 = x1 + np.random.randn(m_col) * 0.01  # Nearly identical to x1!
x3 = np.random.randn(m_col)

X_collinear = np.column_stack([x1, x2, x3])
y_collinear = 3 * x1 + 2 * x3 + np.random.randn(m_col) * 0.5

# OLS
lr_col = LinearRegression().fit(X_collinear, y_collinear)
# Ridge
ridge_col = Ridge(alpha=1.0).fit(X_collinear, y_collinear)

print('With highly correlated features (x1 ≈ x2):')
print(f'  Correlation between x1 and x2: {np.corrcoef(x1, x2)[0,1]:.6f}')
print(f'')
print(f'  OLS coefficients:   [{lr_col.coef_[0]:8.2f}, {lr_col.coef_[1]:8.2f}, {lr_col.coef_[2]:8.2f}]')
print(f'  Ridge coefficients: [{ridge_col.coef_[0]:8.2f}, {ridge_col.coef_[1]:8.2f}, {ridge_col.coef_[2]:8.2f}]')
print(f'  True relationship:  y = 3*x1 + 0*x2 + 2*x3')
print(f'')
print('Notice: OLS splits the effect of x1 wildly between x1 and x2.')
print('Ridge distributes the effect more evenly and stably.')

---
## 🏆 Section 7: Challenge Solution

In [ ]:
# ✅ SOLUTION: Feature selection challenge
np.random.seed(42)
n = 300

hours_studied = np.random.uniform(1, 10, n)
practice_tests = np.random.randint(0, 20, n).astype(float)
sleep_hours = np.random.uniform(4, 10, n)
attendance_pct = np.random.uniform(50, 100, n)
prev_gpa = np.random.uniform(2.0, 4.0, n)
noise_features = np.random.randn(n, 10)

exam_score = (5 * hours_studied + 2 * practice_tests + 3 * sleep_hours 
              + 0.5 * attendance_pct + 10 * prev_gpa 
              + np.random.randn(n) * 3)

feature_names = (['hours', 'practice', 'sleep', 'attend', 'gpa'] + 
                 [f'noise_{i}' for i in range(10)])
X_exam = np.column_stack([hours_studied, practice_tests, sleep_hours, 
                          attendance_pct, prev_gpa, noise_features])

# Step 1: Split
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_exam, exam_score, test_size=0.2, random_state=42)

# Step 2: Scale
sc = StandardScaler()
X_tr_s = sc.fit_transform(X_tr)
X_te_s = sc.transform(X_te)

In [ ]:
# Step 3: Fit all models
challenge_models = {
    'Linear': LinearRegression(),
    'RidgeCV': RidgeCV(alphas=np.logspace(-4, 4, 50), cv=5),
    'LassoCV': LassoCV(cv=5, random_state=42),
    'ElasticNetCV': ElasticNetCV(cv=5, l1_ratio=[0.1, 0.5, 0.9], random_state=42)
}

results = {}
for name, model in challenge_models.items():
    model.fit(X_tr_s, y_tr)
    y_pred = model.predict(X_te_s)
    results[name] = {
        'r2': r2_score(y_te, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_te, y_pred)),
        'coefs': model.coef_,
        'n_zeros': sum(abs(c) < 1e-6 for c in model.coef_)
    }

# Print summary
print(f'{"Model":<16} {"R²":>8} {"RMSE":>8} {"# Zeros":>8}')
print('-' * 45)
for name, r in results.items():
    print(f'{name:<16} {r["r2"]:8.4f} {r["rmse"]:8.4f} {r["n_zeros"]:8d}')

In [ ]:
# Step 4: Coefficient bar charts
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (name, r) in zip(axes.ravel(), results.items()):
    colors = ['#2ecc71' if i < 5 else '#e74c3c' for i in range(15)]
    bars = ax.bar(range(15), np.abs(r['coefs']), color=colors, edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(15))
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('|Coefficient|')
    ax.set_title(f'{name}  (R²={r["r2"]:.4f}, zeros={r["n_zeros"]})', fontsize=12)
    ax.axhline(y=0, color='k', linewidth=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#2ecc71', label='Relevant features'),
                   Patch(facecolor='#e74c3c', label='Noise features')]
fig.legend(handles=legend_elements, loc='upper center', ncol=2, fontsize=12)

plt.suptitle('Feature Selection: Which Model Identifies Relevant Features?', fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

print('\n✅ Lasso/ElasticNet correctly zero out noise features!')
print('Ridge shrinks noise features but doesn\'t eliminate them.')
print('Linear regression gives non-zero (noisy) coefficients to ALL features.')

---
## ✅ Summary

Key takeaways:
- Ridge: shrinks all coefficients, handles multicollinearity, has closed-form
- Lasso: zeros out irrelevant features, automatic feature selection
- ElasticNet: best of both, handles correlated features
- Always standardize before regularization!
- Use CV (RidgeCV, LassoCV) for automatic alpha selection

**🎉 Phase 1 Complete!**  
**Next Phase**: [Classification](../../classification/) →